# Feature Engineering Walkthrough

This notebook explains and runs the preprocessing and feature engineering pipeline for the CIC-IDS2017 network intrusion detection project.

Input: `hdfs://namenode:9000/user/bigdata/ids2017/processed/cleaned`

Output: `hdfs://namenode:9000/user/bigdata/ids2017/processed/ml_ready_binary`

The final output contains `features`, `label`, and `label_original`, ready for Spark MLlib modeling.

## 1. Import PySpark libraries

This cell imports Spark SQL functions, numeric type detection, and `VectorAssembler`, which is required to create the final `features` vector for Spark MLlib.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import NumericType
from pyspark.ml.feature import VectorAssembler


## 2. Define HDFS input and output paths

This cell defines where the notebook reads the cleaned Parquet data from and where it writes the final ML-ready dataset.

In [ ]:
# -------------------------------------------------------
# Paths
# -------------------------------------------------------
HDFS_CLEANED = "hdfs://namenode:9000/user/bigdata/ids2017/processed/cleaned"
HDFS_ML_READY = "hdfs://namenode:9000/user/bigdata/ids2017/processed/ml_ready_binary"


## 3. Start the Spark session

This cell connects the notebook to the Spark standalone cluster through `spark://spark-master:7077` and sets memory/partition options suitable for the Docker environment.

In [ ]:
# -------------------------------------------------------
# Spark Session
# -------------------------------------------------------
spark = (
    SparkSession.builder
    .appName("IDS2017-FeatureEngineering")
    .master("spark://spark-master:7077")
    .config("spark.executor.memory", "1500m")
    .config("spark.executor.cores", "1")
    .config("spark.driver.memory", "512m")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

## 4. Read cleaned Parquet data

This cell reads the cleaned Parquet dataset, then prints the row count, column count, and schema.

In [ ]:
# -------------------------------------------------------
# Step 1: Read cleaned data
# -------------------------------------------------------
print("Step 1: Reading cleaned Parquet")

df = spark.read.parquet(HDFS_CLEANED)

print("Rows:", df.count())
print("Columns:", len(df.columns))
df.printSchema()

## 5. Check original multiclass label distribution

This cell shows the original labels such as `BENIGN`, `DOS_HULK`, `PORT_SCAN`, and other attack types.

In [ ]:
# -------------------------------------------------------
# Step 2: Check original labels
# -------------------------------------------------------
print("Step 2: Original label distribution")

df.groupBy("label").count().orderBy(F.desc("count")).show(50, truncate=False)


## 6. Convert original labels to binary labels

This cell keeps the original attack name in `label_original`, then creates a binary target label where `BENIGN = 0` and all attack types equal `1`.

In [ ]:
# -------------------------------------------------------
# Step 3: Convert labels to binary
# BENIGN = 0
# Any attack = 1
# -------------------------------------------------------
print("Step 3: Creating binary label")

df = df.withColumnRenamed("label", "label_original")

df = df.withColumn(
    "label",
    F.when(F.col("label_original") == "BENIGN", F.lit(0.0)).otherwise(F.lit(1.0))
)

print("Binary label distribution:")
df.groupBy("label").count().orderBy("label").show()


## 7. Create engineered network-flow features

This cell creates extra features such as total packets, total payload bytes, average payload per packet, forward/backward packet ratio, and log-transformed features.

In [ ]:
# -------------------------------------------------------
# Step 4: Feature Engineering
# -------------------------------------------------------
print("Step 4: Creating engineered features")

# total packets = forward packets + backward packets
if "fwd_packets_count" in df.columns and "bwd_packets_count" in df.columns:
    df = df.withColumn(
        "total_packets",
        F.col("fwd_packets_count") + F.col("bwd_packets_count")
    )

# total payload bytes = forward payload + backward payload
if "fwd_total_payload_bytes" in df.columns and "bwd_total_payload_bytes" in df.columns:
    df = df.withColumn(
        "total_payload_bytes",
        F.col("fwd_total_payload_bytes") + F.col("bwd_total_payload_bytes")
    )

# average payload per packet
if "total_payload_bytes" in df.columns and "total_packets" in df.columns:
    df = df.withColumn(
        "avg_payload_per_packet",
        F.when(
            F.col("total_packets") > 0,
            F.col("total_payload_bytes") / F.col("total_packets")
        ).otherwise(F.lit(0.0))
    )

# ratio between forward and backward packets
if "fwd_packets_count" in df.columns and "bwd_packets_count" in df.columns:
    df = df.withColumn(
        "fwd_bwd_packet_ratio",
        F.when(
            F.col("bwd_packets_count") > 0,
            F.col("fwd_packets_count") / F.col("bwd_packets_count")
        ).otherwise(F.lit(0.0))
    )

# log-transformed features to reduce impact of very large values
if "duration" in df.columns:
    df = df.withColumn(
        "log_duration",
        F.log1p(F.when(F.col("duration") >= 0, F.col("duration")).otherwise(F.lit(0.0)))
    )

if "bytes_rate" in df.columns:
    df = df.withColumn(
        "log_bytes_rate",
        F.log1p(F.when(F.col("bytes_rate") >= 0, F.col("bytes_rate")).otherwise(F.lit(0.0)))
    )

if "packets_rate" in df.columns:
    df = df.withColumn(
        "log_packets_rate",
        F.log1p(F.when(F.col("packets_rate") >= 0, F.col("packets_rate")).otherwise(F.lit(0.0)))
    )


## 8. Select numeric feature columns

This cell selects all numeric feature columns using `NumericType`, excluding `label` and `label_original`, then casts them to double for MLlib compatibility.

In [ ]:
# -------------------------------------------------------
# Step 5: Detect numeric feature columns
# Use NumericType so we include Double, Integer, Long, etc.
# -------------------------------------------------------
print("Step 5: Selecting numeric feature columns")

ignore_cols = {"label", "label_original"}

numeric_cols = [
    field.name
    for field in df.schema.fields
    if field.name not in ignore_cols and isinstance(field.dataType, NumericType)
]

print("Number of numeric feature columns:", len(numeric_cols))

if len(numeric_cols) == 0:
    raise ValueError("No numeric feature columns found. Check schema.")

# Cast all numeric columns to double for Spark MLlib consistency
for c in numeric_cols:
    df = df.withColumn(c, F.col(c).cast("double"))


## 9. Convert NaN values to null

This cell converts `NaN` values into null values so they can be handled consistently in the next missing-value step.

In [ ]:
# -------------------------------------------------------
# Step 6: Convert NaN values to null
# -------------------------------------------------------
print("Step 6: Converting NaN values to null")

for c in numeric_cols:
    df = df.withColumn(
        c,
        F.when(F.isnan(F.col(c)), None).otherwise(F.col(c))
    )


## 10. Handle outliers using percentile capping

This cell caps numeric values below the 1st percentile and above the 99th percentile. This reduces extreme numeric effects without deleting attack records.

In [ ]:
# -------------------------------------------------------
# Step 7: Outlier handling using percentile capping
# Cap values below 1st percentile and above 99th percentile
# -------------------------------------------------------
print("Step 7: Handling outliers using percentile capping")

LOWER_Q = 0.01
UPPER_Q = 0.99
REL_ERROR = 0.01

quantiles = df.approxQuantile(numeric_cols, [LOWER_Q, UPPER_Q], REL_ERROR)

for col_name, bounds in zip(numeric_cols, quantiles):
    if len(bounds) == 2:
        lower, upper = bounds

        # Skip invalid or constant columns
        if lower is not None and upper is not None and lower < upper:
            df = df.withColumn(
                col_name,
                F.when(F.col(col_name) < lower, F.lit(lower))
                 .when(F.col(col_name) > upper, F.lit(upper))
                 .otherwise(F.col(col_name))
            )

print("Outlier capping completed")


## 11. Handle remaining missing values

This cell fills remaining numeric null values with `0.0`, ensuring the feature vector can be created without missing values.

In [ ]:
# -------------------------------------------------------
# Step 8: Handle missing values after outlier capping
# -------------------------------------------------------
print("Step 8: Handling missing values")

df = df.fillna(0.0, subset=numeric_cols)


## 12. Validate important feature columns

This cell prints summary statistics for important engineered and original columns after preprocessing.

In [ ]:
# -------------------------------------------------------
# Step 9: Validate important columns after preprocessing
# -------------------------------------------------------
print("Step 9: Validation summary for important columns")

important_cols = [
    "duration",
    "bytes_rate",
    "packets_rate",
    "total_packets",
    "total_payload_bytes",
    "avg_payload_per_packet",
    "fwd_bwd_packet_ratio",
    "log_duration",
    "log_bytes_rate",
    "log_packets_rate"
]

existing_important_cols = [c for c in important_cols if c in df.columns]

if existing_important_cols:
    df.select(existing_important_cols).summary(
        "count", "min", "max", "mean", "stddev"
    ).show(truncate=False)


## 13. Assemble Spark MLlib feature vector

This cell combines all numeric input columns into one vector column named `features`, which Spark MLlib models require.

In [ ]:
# -------------------------------------------------------
# Step 10: Assemble features vector for Spark MLlib
# -------------------------------------------------------
print("Step 10: Assembling features vector")

assembler = VectorAssembler(
    inputCols=numeric_cols,
    outputCol="features",
    handleInvalid="keep"
)

df_ml = assembler.transform(df).select(
    "features",
    "label",
    "label_original"
)


## 14. Validate final ML-ready dataset

This cell verifies the final schema, row count, binary label distribution, and original label distribution.

In [ ]:
# -------------------------------------------------------
# Step 11: Final validation
# -------------------------------------------------------
print("Step 11: Final ML-ready dataset validation")

df_ml.printSchema()

print("Final row count:", df_ml.count())

print("Final binary label distribution:")
df_ml.groupBy("label").count().orderBy("label").show()

print("Original labels still available:")
df_ml.groupBy("label_original").count().orderBy(F.desc("count")).show(50, truncate=False)


## 15. Save ML-ready dataset to HDFS

This cell writes the final output as Parquet to HDFS, ready for Spark MLlib modeling.

In [ ]:
# -------------------------------------------------------
# Step 12: Save final ML-ready dataset
# -------------------------------------------------------
print("Step 12: Writing ML-ready dataset to HDFS")

df_ml.write.mode("overwrite").parquet(HDFS_ML_READY)

print("Done.")
print("Output saved to:")
print(HDFS_ML_READY)


## 16. Stop Spark

This cell stops the Spark session after the notebook finishes.

In [ ]:
spark.stop()
